# VRP 03 — Passive execution of the ATM straddle (both sides)

**Discussion:** `discussions/Volatility_Risk_Premium/009_claude_execution_and_conditional_short.md` · **Code:** `vrp_execution.py`

Can a limit order near the mid recover the spread that killed VRP 02 (buy at ask −27%, mid bound −14%)? Same hourly ATM entries as VRP 02.

- Each leg rests at $P_b=\text{mid}-q(\text{mid}-\text{bid})$ (buy) or $P_s=\text{mid}+q(\text{ask}-\text{mid})$ (sell) for $W$ minutes; an unfilled leg is crossed at $t+W$ ("passive-then-cross").
- **Fill proxy on 5m snapshots:** the opposite touch reaches $P$, or a trade prints at or through $P$ (volume or last price changed). Optimistic at this grid.
- **Primary:** $q=0.5$, $W=15$ min. Paired $\Delta R$ = policy − aggressive per entry; session bootstrap.
- Verdict per side: `PASSIVE_HELPS` / `PASSIVE_HURTS` / `INCONCLUSIVE`.

Environment: `VRP03_OFFLINE=1` (cached delivery prices only), `VRP03_N_BOOT`.

In [ ]:
from pathlib import Path
import json, os, sys, time, warnings
import numpy as np
import pandas as pd
import duckdb
from dotenv import load_dotenv

warnings.filterwarnings('ignore', category=RuntimeWarning)
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 60)
load_dotenv()
DATA_ROOT = Path(os.environ['DERIBIT_DATA_ROOT']).expanduser().resolve()
DB_PATH = DATA_ROOT / 'database' / 'research.duckdb'
PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'src' / 'data_engineering').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'src' / 'data_engineering').exists():
    raise RuntimeError('Repository root not found.')
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from data_engineering.modeling import vrp_long_straddle as ls
VRP = PROJECT_ROOT / 'notebooks' / 'Volatility_Risk_Premium'
from data_engineering.modeling import vrp_execution as ve
OFFLINE = os.getenv('VRP03_OFFLINE', '').strip() == '1'
N_BOOT = int(os.getenv('VRP03_N_BOOT', str(ve.N_BOOT)))
OUT = VRP / '03'
OUT.mkdir(parents=True, exist_ok=True)
print('VRP 03', ve.VERSION, ve.selftest())
print(json.dumps(ve.registration(), indent=2))
T0 = time.time()

In [ ]:
def load_chain(cols_wanted, where_extra, hourly):
    con = duckdb.connect(str(DB_PATH), read_only=True)
    try:
        have = set(con.execute('DESCRIBE normalized_option_chain').df()['column_name'])
        need = [c for c in ls.CHAIN_COLUMNS if c != 'settlement_currency']
        miss = sorted(set(need) - have)
        if miss:
            raise RuntimeError(f'normalized_option_chain lacks {miss}')
        cols = [c for c in cols_wanted if c in have]
        grid = f'AND EXTRACT(minute FROM n.state_time) % {ls.ENTRY_GRID_MIN} = 0' if hourly else ''
        df = con.execute(f'''
            WITH prim AS (SELECT DISTINCT state_id, option_family_id FROM primary_option_surface_feature_state)
            SELECT {', '.join('n.' + c for c in cols)}
            FROM normalized_option_chain n
            JOIN prim p ON p.state_id = n.state_id AND p.option_family_id = n.option_family_id
            WHERE n.minutes_to_expiry BETWEEN 60 AND {ls.MAX_TTE_MIN} {grid} {where_extra}
        ''').df()
    finally:
        con.close()
    for c in cols_wanted:
        if c not in df:
            df[c] = None if c == 'settlement_currency' else np.nan
    return df

def settlement(st, cache_out, offline):
    sessions = sorted(st.session.unique())
    con = duckdb.connect(str(DB_PATH), read_only=True)
    try:
        idx = con.execute('SELECT state_time, underlying, futures_index_price FROM time_series_feature_state').df()
    finally:
        con.close()
    approx = ls.approx_settlement(idx, sessions)
    names = sorted({ls.index_name(u, c) for u, c in zip(st.underlying, st.settlement_currency)})
    seed = VRP / '02' / '02_delivery_prices.csv'
    if not cache_out.exists() and seed.exists():
        cache_out.write_text(seed.read_text())
    if offline:
        delivery = pd.read_csv(cache_out, dtype={'session': str}) if cache_out.exists() else pd.DataFrame(columns=['index_name', 'session', 'delivery_price'])
    else:
        delivery = ls.fetch_delivery_prices(names, cache_csv=cache_out)
    delivery = delivery[delivery.session.isin(sessions)]
    x = ls.attach_settlement(st, delivery, approx)
    print('settlement source:', x.drop_duplicates(['session', 'underlying']).settle_source.value_counts().to_dict())
    return x


## 1. Snapshots, ticks, entries

In [ ]:
chain = load_chain(list(ve.CHAIN_COLUMNS), f"AND abs(ln(n.strike / n.option_forward_price)) < 0.05 AND n.state_time < TIMESTAMPTZ '{ls.VRP_CONFIRMATION_START.isoformat()}'", hourly=False)
print(f'chain rows {len(chain):,} | trade fields present: last_price={chain.last_price.notna().any()}, volume_24h={chain.volume_24h.notna().any()}')
snap = ve.prepare_snapshots(chain)
del chain
ticks = ve.family_ticks(snap)
display(ticks.to_frame())
st = ve.straddle_entries(snap)
st = settlement(st, OUT / '03_delivery_prices.csv', OFFLINE)
st = st[np.isfinite(st.S_T)].reset_index(drop=True)
print(f'entries {len(st):,} | sessions {st.session.nunique()} | {time.time()-T0:.0f}s')

## 2. Policy grid and primary decision

In [ ]:
res = ve.evaluate(st, snap, ticks, n_boot=N_BOOT)
grid = res['grid']
display(grid.round(4))
print(json.dumps(res['decision'], indent=2, default=float))

## 3. Primary configuration by TTE bucket and asset (reported)

In [ ]:
rows = []
for side, pe in res['per_entry'].items():
    for by in ('tte_bucket', 'underlying'):
        for lvl, g in pe.groupby(by):
            d = ve.session_boot(g.delta.to_numpy(), g.session.to_numpy(), 1000)
            a = ve.session_boot(g.R_policy.to_numpy(), g.session.to_numpy(), 1000)
            rows.append({'side': side, 'group': by, 'level': lvl, 'n': len(g), 'fill_both': g.both_filled.mean(),
                         'delta_M': d.get('M'), 'delta_ci_low': d.get('ci_low'), 'delta_ci_high': d.get('ci_high'),
                         'R_policy_M': a.get('M'), 'R_ci_low': a.get('ci_low'), 'R_ci_high': a.get('ci_high')})
by = pd.DataFrame(rows)
display(by.round(4))

## 4. Outputs

In [ ]:
grid.to_csv(OUT / '03_policy_grid.csv', index=False, float_format='%.6g')
by.to_csv(OUT / '03_primary_by_group.csv', index=False, float_format='%.6g')
ticks.to_frame().to_csv(OUT / '03_ticks.csv')
(OUT / '03_decision.json').write_text(json.dumps({'decision': res['decision'], 'registration': ve.registration()}, indent=2, default=float))
meta = {'module': '03_passive_execution', 'version': ve.VERSION, 'offline': OFFLINE, 'n_entries': int(len(st)),
        'sessions': [st.session.min(), st.session.max()], 'n_sessions': int(st.session.nunique()),
        'settle_sources': st.drop_duplicates(['session', 'underlying']).settle_source.value_counts().to_dict(),
        'runtime_s': round(time.time() - T0)}
(OUT / '03_run_metadata.json').write_text(json.dumps(meta, indent=2, default=str))
print(json.dumps(meta, indent=2, default=str))